# Lecture 3.5 — User Interaction: AskUserQuestion
## Section 3 — Built-In Tools Deep Dive
### Course: Claude Agent SDK: Build AI Agents in Python

---

In this notebook you will build a **human-in-the-loop agent** — one that pauses mid-task, asks you a clarifying question with multiple choice options, and continues based on your answer.

You will build:
- A **streaming async generator** prompt — signals an interactive session to the SDK
- A **dummy `PreToolUse` hook** — sends keep-alive heartbeats to maintain the HTTP connection
- A **`can_use_tool` callback** — intercepts every tool call and routes by `tool_name`
- A **complete interactive agent** that asks clarifying questions before giving advice

This notebook is self-contained. It does not depend on any previous lecture's notebook.

In [11]:
# Cell 2 — Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

In [2]:
# Cell 3 — Load API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

We use a single `MODEL_NAME` variable throughout this notebook.  
Change it here to switch between Claude models without editing each cell.

For the latest available models, visit:  
https://platform.claude.com/docs/en/about-claude/models/overview

In [3]:
# Cell 5 — Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

---

## Part 1 — The Streaming Async Generator Pattern

In every lecture so far we passed a **plain string** to `query()`.  
With `AskUserQuestion` we must pass an **async generator** instead.

This is not just a syntax change — it is a **signal to the SDK**:
> *"This is an interactive session. Keep the HTTP connection open. More messages may be coming."*

Without this signal, the SDK expects one prompt, one response, and connection closure.  
It would not know to wait while a human types their answer.

The content inside the generator — the `content` field — is exactly the same prompt  
you have been writing all along. Only the wrapper changes.

In [4]:
# Cell 7 — The prompt_stream async generator
# This wrapper signals an interactive session to the SDK
async def prompt_stream():
    yield {
        "type": "user",
        "message": {
            "role": "user",
            "content": "Your prompt here"  # <-- same prompt as always, just wrapped
        }
    }

---

## Part 2 — The Dummy PreToolUse Hook

When the `can_use_tool` callback is waiting for the user to type, the HTTP connection  
needs periodic keep-alive signals to stay open. Without them, the connection may time out.

The `PreToolUse` hook fires **before each tool use**. Our dummy version returns  
`{"continue_": True}` — that return value is the keep-alive signal.

**Execution order:** Hook fires first → Callback fires → Tool executes

Think of it as a **heartbeat**: the HTTP connection needs a regular pulse to stay alive.  
The dummy hook provides that pulse — once before each tool use.

In [5]:
# Cell 9 — The dummy PreToolUse hook
# Returns {"continue_": True} — the keep-alive signal for the HTTP connection
async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}

---

## Part 3 — The AskUserQuestion Input Structure

When `tool_name == 'AskUserQuestion'`, the `input_data` argument contains a `questions` array.  
Each question has these fields:

| Field | Type | Description |
|---|---|---|
| `question` | str | The actual question Claude wants to ask |
| `header` | str | A short category label for the question |
| `options` | list | Each option has `label` and `description` |
| `multiSelect` | bool | Whether the user can select more than one option |

**Claude generates these questions and options** — you cannot inject your own.  
Your job is to display them clearly and collect the user's answer.

In [6]:
# Cell 11 — Show the AskUserQuestion input_data structure
import json

# This is exactly what input_data contains when AskUserQuestion fires
# Claude generates this — you do not write the questions
example_input_data = {
    "questions": [
        {
            "question": "How should I format the output?",
            "header": "Format",
            "options": [
                {"label": "Summary", "description": "Brief overview"},
                {"label": "Detailed", "description": "Full explanation"}
            ],
            "multiSelect": False
        }
    ]
}

print(json.dumps(example_input_data, indent=2))

{
  "questions": [
    {
      "question": "How should I format the output?",
      "header": "Format",
      "options": [
        {
          "label": "Summary",
          "description": "Brief overview"
        },
        {
          "label": "Detailed",
          "description": "Full explanation"
        }
      ],
      "multiSelect": false
    }
  ]
}


---

## Hooks vs Callbacks — A Critical Distinction

There are two objects firing when Claude uses a tool. They look similar but serve completely different purposes.

### `can_use_tool` — Callback
- Makes **allow / deny decisions** — Claude waits for your return value before proceeding
- **Pauses execution** while waiting — this is where `input()` happens
- Fires for every tool call — you route by `tool_name`
- Returns `PermissionResultAllow` or `PermissionResultDeny`

### `PreToolUse` dummy hook — Hook
- **Observes and signals** — does not make allow/deny decisions
- Does **not** pause the main execution flow
- Returns `{"continue_": True}` — a keep-alive signal to the SDK
- Fires before each tool use

**Order: Hook fires first → Callback fires → Tool executes (every single time)**

The hook keeps the HTTP connection alive.  
The callback decides what happens next.

---

## Part 4 — Building the Handler: `parse_response` Helper

Users might type their answer in different ways:
- A number like `1` to select the first option
- Comma-separated numbers like `1, 3` for multi-select questions
- The label text directly like `Summary`

`parse_response` converts their raw typed input to the option label(s)  
that Claude expects in the `answers` dictionary.

In [7]:
# Cell 14 — parse_response helper
def parse_response(response: str, options: list) -> str:
    try:
        # Try to interpret as number(s) — users type 1-based, Python lists are 0-based
        indices = [int(s.strip()) - 1 for s in response.split(",")]
        labels = [options[i]["label"] for i in indices if 0 <= i < len(options)]
        return ", ".join(labels) if labels else response
    except ValueError:
        # User typed the label directly — return it as-is
        return response

---

## Part 5 — Building the Handler: `handle_ask_user_question`

This function is called by the `can_use_tool` callback whenever `tool_name == 'AskUserQuestion'`.  
It does the following:

1. Loops through every question Claude generated
2. Prints the header and question text
3. Prints numbered options
4. Calls `input()` — **execution pauses here** while the user types
5. Calls `parse_response` to convert their input to an option label
6. Returns `PermissionResultAllow` with both the original questions and the collected answers

The key detail: we return **both** `questions` and `answers` inside `updated_input`.  
The SDK packages this as a **ToolResponse** and sends it back to Claude through the open HTTP connection.

In [8]:
# Cell 16 — handle_ask_user_question
from claude_agent_sdk.types import PermissionResultAllow


async def handle_ask_user_question(input_data: dict):
    answers = {}

    for q in input_data.get("questions", []):
        # Print the header and the question Claude generated
        print(f"\n{q['header']}: {q['question']}")

        # Print each option with a 1-based number for easy selection
        options = q["options"]
        for i, opt in enumerate(options):
            print(f"  {i + 1}. {opt['label']} - {opt['description']}")

        # Tell the user how to answer based on whether multi-select is allowed
        if q.get("multiSelect"):
            print("  (Enter numbers separated by commas, or type your own answer)")
        else:
            print("  (Enter a number, or type your own answer)")

        # *** PAUSE — execution stops here while the user types ***
        response = input("Your choice: ").strip()
        answers[q["question"]] = parse_response(response, options)

    # Return both questions and answers — the SDK sends this as a ToolResponse to Claude
    return PermissionResultAllow(
        updated_input={
            "questions": input_data.get("questions", []),
            "answers": answers,
        }
    )

---

## Part 6 — The `can_use_tool` Callback

This is the routing function. It fires for **every** tool call Claude makes — not just `AskUserQuestion`.

- If `tool_name == 'AskUserQuestion'` → call our handler (pause + ask user)
- For all other tools → auto-approve and let them execute normally

Think of it as a **security guard at a door** — stops everyone, decides what to do based on who it is.  
Most visitors get waved through. One type — `AskUserQuestion` — gets special treatment.

In [9]:
# Cell 18 — can_use_tool callback
async def can_use_tool(tool_name: str, input_data: dict, context):
    if tool_name == "AskUserQuestion":
        # Handle specially — display questions, collect answers, return as ToolResponse
        return await handle_ask_user_question(input_data)
    # For any other tool (Read, Glob, Bash, etc.): auto-approve and let it execute
    return PermissionResultAllow(updated_input=input_data)

---

## Part 7 — Putting It All Together

The cell below wires every piece we've built into a complete runnable agent.

### Critical: Why the prompt must be explicit

`AskUserQuestion` is not automatically called whenever Claude wants to ask something.  
Claude can also write questions as plain text — and that is its **default behaviour**.  
If the prompt only says *"ask me clarifying questions"*, Claude will write formatted  
markdown questions instead of calling the tool. The `can_use_tool` callback will never fire.

The prompt must explicitly say:
- `You MUST use the AskUserQuestion tool to ask me clarifying questions`
- `Do NOT write questions as plain text — only use the AskUserQuestion tool`

### What to expect when you run Cell 20

1. Claude receives the prompt and starts working
2. Claude calls `AskUserQuestion` — the `can_use_tool` callback fires
3. **The cell will pause** — you'll see a question printed and a `Your choice:` prompt
4. Type a number (e.g. `1`) and press Enter for each question
5. Claude receives your answers and returns a recommendation tailored to what you chose

> ⚠️ **If the cell seems to hang — it hasn't.** It is waiting for your input.  
> Look for the `Your choice:` prompt in the output area and type your answer.

In [10]:
# Cell 20 — Full combined runnable example
import asyncio
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query
from claude_agent_sdk.types import HookMatcher, PermissionResultAllow


# ── Helper: convert typed input to option label ────────────────────────────────
def parse_response(response: str, options: list) -> str:
    try:
        indices = [int(s.strip()) - 1 for s in response.split(",")]
        labels = [options[i]["label"] for i in indices if 0 <= i < len(options)]
        return ", ".join(labels) if labels else response
    except ValueError:
        return response


# ── Handler: display questions, collect answers, return as ToolResponse ─────────
async def handle_ask_user_question(input_data: dict):
    answers = {}

    for q in input_data.get("questions", []):
        print(f"\n{q['header']}: {q['question']}")

        options = q["options"]
        for i, opt in enumerate(options):
            print(f"  {i + 1}. {opt['label']} - {opt['description']}")
        if q.get("multiSelect"):
            print("  (Enter numbers separated by commas, or type your own answer)")
        else:
            print("  (Enter a number, or type your own answer)")

        response = input("Your choice: ").strip()
        answers[q["question"]] = parse_response(response, options)

    return PermissionResultAllow(
        updated_input={
            "questions": input_data.get("questions", []),
            "answers": answers,
        }
    )


# ── Routing callback: fires for every tool, routes AskUserQuestion specially ───
async def can_use_tool(tool_name: str, input_data: dict, context):
    if tool_name == "AskUserQuestion":
        return await handle_ask_user_question(input_data)
    return PermissionResultAllow(updated_input=input_data)


# ── Dummy hook: keep-alive heartbeat for the HTTP connection ──────────────────
async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}


# ── Streaming prompt: signals an interactive session to the SDK ────────────────
# IMPORTANT: The prompt must explicitly tell Claude to use the AskUserQuestion
# tool. Without this, Claude will write questions as plain text instead of
# calling the tool — and the can_use_tool callback will never fire.
async def prompt_stream():
    yield {
        "type": "user",
        "message": {
            "role": "user",
            "content": """
            Help me decide on the tech stack for a new mobile app.
            You MUST use the AskUserQuestion tool to ask me clarifying questions.
            Do NOT write questions as plain text — only use the AskUserQuestion tool.
            Ask about:
            1. What platform I am targeting
            2. What my team's experience level is
            Then give your recommendation based on my answers.
            """,
        },
    }


# ── Run the agent ──────────────────────────────────────────────────────────────
async for message in query(
    prompt=prompt_stream(),
    options=ClaudeAgentOptions(
        allowed_tools=["AskUserQuestion"],
        can_use_tool=can_use_tool,
        model=MODEL_NAME,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher=None, hooks=[dummy_hook])
            ]
        },
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)


Platform: Which platform(s) are you targeting for your mobile app?
  1. iOS only - App will only be available on Apple's iOS platform
  2. Android only - App will only be available on Google's Android platform
  3. Both iOS and Android (native) - Native apps for both platforms using separate codebases
  4. Cross-platform (single codebase) - One codebase that works on both iOS and Android
  (Enter numbers separated by commas, or type your own answer)
Your choice: 4

Team Experience: What is your team's experience level with mobile development?
  1. Beginners - Limited to no experience with mobile app development
  2. Intermediate - Some experience with mobile development, familiar with basic concepts
  3. Advanced - Extensive experience with mobile development, comfortable with complex architectures
  (Enter a number, or type your own answer)
Your choice: 1
Perfect! Based on your answers, here's my tech stack recommendation for your cross-platform mobile app with a beginner-level team:

---

## Limitations Worth Knowing

**Not available in subagents**  
If your agent spawns subagents (covered in Section 7), those subagents cannot call `AskUserQuestion`.  
Only the top-level agent can.

**Question and option count limits**  
Each call supports 1–4 questions, and each question supports 2–4 options.

**Claude generates the questions**  
You cannot inject your own questions into this flow. If you need to ask the user something  
before the agent starts, do it in your application code before calling `query()`.

**Don't overuse it**  
The value of an agent is automation. If the agent asks questions at every step, it defeats  
the purpose. Reserve `AskUserQuestion` for genuine decision points where the user's choice  
meaningfully changes the outcome.

---

## Summary — Lecture 3.5

**What you learned:**

- **Human-in-the-loop pattern** — agent pauses mid-task, asks the user, continues with the answer
- **HTTP connection** — stays open during the pause; streaming input signals an interactive session
- **Hook vs callback distinction** — hook observes/signals (fires first, keep-alive), callback decides (fires second, routes by `tool_name`)
- **`AskUserQuestion` is unique** — no automatic execution; you provide the answer in `updated_input` as a `ToolResponse`
- **Five implementation pieces** — `prompt_stream`, `dummy_hook`, `parse_response`, `handle_ask_user_question`, `can_use_tool`
- **Limitations** — not in subagents, 1–4 questions, 2–4 options, Claude generates them, don't overuse

---

**Coming up in Lecture 3.6 — Hands-On: Codebase TODO Summarizer Agent**  
You will scan a real codebase for TODO comments using Glob and Grep, categorize them,  
and write a clean structured summary report — the Section 3 capstone project.